# Component 1 — Persistent KV Cache Manager (LMCache)
AgentCache · Manav Patel, Yash Malegaonkar, Danyal Khan

Runs vLLM + LMCache on Colab T4. Measures TTFT across cold / warm / hot cache states for a coding agent and a search agent under round-robin switching.

## 1 — GPU check

In [1]:
!nvidia-smi

Mon May 18 23:53:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             28W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2 — Install dependencies

In [2]:
!pip install vllm lmcache openai tqdm python-dotenv -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print('done')

done


In [3]:
!which lmcache_server
!ls /usr/local/bin/ | grep -i lmcache
!find /usr/local/lib/python3.12/dist-packages/lmcache -name "*.py" -path "*/server*"

/usr/local/bin/lmcache_server
lmcache
lmcache_controller
lmcache_server
/usr/local/lib/python3.12/dist-packages/lmcache/cli/commands/server.py
/usr/local/lib/python3.12/dist-packages/lmcache/v1/server/storage_backend/__init__.py
/usr/local/lib/python3.12/dist-packages/lmcache/v1/server/storage_backend/local_backend.py
/usr/local/lib/python3.12/dist-packages/lmcache/v1/server/storage_backend/abstract_backend.py
/usr/local/lib/python3.12/dist-packages/lmcache/v1/server/__init__.py
/usr/local/lib/python3.12/dist-packages/lmcache/v1/server/__main__.py
/usr/local/lib/python3.12/dist-packages/lmcache/v1/server/utils.py
/usr/local/lib/python3.12/dist-packages/lmcache/v1/multiprocess/server.py


## 3 — Environment + LMCache config

In [4]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.makedirs('/content/agentcache/kv_store', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)

with open('/content/lmcache_disk.yaml', 'w') as f:
    f.write(
        'chunk_size: 256\n'
        'local_cpu: true\n'
        'max_local_cpu_size: 4.0\n'
        'eviction_policy: LRU\n'
    )

print('config written')
!cat /content/lmcache_disk.yaml

config written
chunk_size: 256
local_cpu: true
max_local_cpu_size: 4.0
eviction_policy: LRU


## 4 — Start vLLM + LMCache server

In [5]:
import subprocess, sys, os, time
import requests

env = os.environ.copy()
env['LMCACHE_CONFIG_FILE'] = '/content/lmcache_disk.yaml'

def start_vllm():
    return subprocess.Popen(
        f"{sys.executable} -m vllm.entrypoints.openai.api_server "
        "--model hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 "
        "--quantization awq "
        "--port 8000 "
        "--max-model-len 4096 "
        "--gpu-memory-utilization 0.90 "
        "--enable-prefix-caching "
        "--kv-offloading-backend lmcache "
        "--kv-offloading-size 4 "
        "--disable-hybrid-kv-cache-manager "
        "> /content/vllm.log 2>&1",
        shell=True, env=env
    )

def wait_for_vllm(timeout_sec=600):
    print('Waiting for vLLM...')
    for i in range(timeout_sec // 5):
        try:
            if requests.get('http://localhost:8000/health').status_code == 200:
                print('vLLM is ready.')
                return True
        except:
            pass
        if i % 6 == 0:
            print(f'  still loading... ({i*5}s elapsed)')
        time.sleep(5)
    print('Timed out — check /content/vllm.log')
    return False

vllm_proc = start_vllm()
print(f'vLLM starting (pid {vllm_proc.pid})...')

vLLM starting (pid 13058)...


In [6]:
print(open('/content/vllm.log').read())

### Watch startup logs (run while waiting)

In [7]:
!tail -30 /content/vllm.log

In [8]:
print(open('/content/vllm.log').read())

In [9]:
print(open('/content/vllm.log').read())

In [10]:
!grep -A 5 "ERROR" /content/vllm.log | head -60

In [11]:
!lmcache_server --help

[2026-05-18 23:53:55,732] LMCache INFO: Using backend: lmcache.c_ops (__init__.py:97:lmcache)
[2026-05-18 23:53:56,861] LMCache ERROR: Usage: /usr/local/bin/lmcache_server <host> <port> <storage>(default:cpu) (__main__.py:155:lmcache.v1.server.__main__)


## 5 — Wait until server is ready

In [12]:
wait_for_vllm()


Waiting for vLLM...
  still loading... (0s elapsed)
  still loading... (30s elapsed)
  still loading... (60s elapsed)
  still loading... (90s elapsed)
  still loading... (120s elapsed)
vLLM is ready.


True

## 6 — Open public tunnel

In [13]:
import threading, re

PUBLIC_URL = None

def _run_tunnel():
    global PUBLIC_URL
    p = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in p.stdout:
        line = line.decode()
        m = re.search(r'https://\S+\.trycloudflare\.com', line)
        if m:
            PUBLIC_URL = m.group()
            print(f'\nPublic URL: {PUBLIC_URL}\n')

threading.Thread(target=_run_tunnel, daemon=True).start()
time.sleep(10)
print('PUBLIC_URL =', PUBLIC_URL)


Public URL: https://contrast-internal-population-cold.trycloudflare.com

PUBLIC_URL = https://contrast-internal-population-cold.trycloudflare.com


## 7 — Agent definitions

In [14]:
CODING_SYSTEM = (
    'You are an expert software engineer specializing in Python, '
    'algorithms, and system design. You debug code, write clean '
    'implementations, and explain technical concepts precisely. '
    'Always reason step by step before writing any code.'
)

SEARCH_SYSTEM = (
    'You are a knowledgeable research assistant. You answer factual '
    'questions clearly and concisely, drawing on your knowledge of '
    'science, technology, history, and current events. Always reason '
    'through your answer and indicate your confidence level.'
)

AGENT_SYSTEMS = {'coding': CODING_SYSTEM, 'search': SEARCH_SYSTEM}

CODING_QUERIES = [
    'Implement a thread-safe LRU cache in Python with O(1) get and put.',
    'Write a Python context manager that retries a block up to N times on exception.',
    'Design a rate limiter class using the token bucket algorithm.',
    'Implement a trie for autocomplete with insert and search methods.',
    'Write a decorator that caches function results with a TTL.',
    'Optimize this O(n^2) solution to find pairs summing to a target value.',
    'Implement consistent hashing for a distributed cache.',
    'Write a generator that streams large CSV files without loading into memory.',
    'Implement BFS and DFS on a graph represented as an adjacency list.',
    'Design a simple pub/sub event system in Python.',
    'Write a function to detect cycles in a linked list.',
    'Implement a min-heap from scratch in Python.',
    'Implement Dijkstra algorithm and explain its time complexity.',
    'Explain the GIL in Python and when it matters for performance.',
    'Implement a simple MapReduce pipeline in Python.',
    'Explain and implement the producer-consumer pattern with asyncio.',
    'Implement a thread pool from scratch using Python threading primitives.',
    'Write a function to serialize and deserialize a binary tree.',
    'Implement a simple key-value store with WAL for crash recovery.',
    'Implement a sliding window rate limiter using Redis sorted sets.',
]

SEARCH_QUERIES = [
    'What are the main architectural differences between transformers and Mamba SSMs?',
    'Explain how RLHF differs from DPO in LLM fine-tuning.',
    'What is the current state of quantum computing for practical applications?',
    'Summarize the key ideas behind retrieval-augmented generation.',
    'How does PagedAttention improve GPU memory efficiency in LLM serving?',
    'What were the main contributions of the Attention is All You Need paper?',
    'Explain the difference between KV cache quantization and weight quantization.',
    'What is speculative decoding and how does it speed up inference?',
    'How does FlashAttention reduce memory usage compared to standard attention?',
    'What are the tradeoffs between beam search and sampling for text generation?',
    'Explain how LoRA fine-tuning works and why it is parameter-efficient.',
    'What is the difference between pre-training and instruction tuning?',
    'Explain the scaling laws for LLMs and what they predict.',
    'What is mixture of experts and how does it affect model capacity vs compute?',
    'What is chain-of-thought prompting and when does it help most?',
    'Explain how vector databases work for semantic search.',
    'What are the main failure modes of RAG systems in production?',
    'How does temperature affect LLM output diversity and quality?',
    'Explain how multi-head attention differs from single-head attention.',
    'What is the purpose of the KV cache in autoregressive generation?',
]

PROMPTS = (
    [{'agent': 'coding', 'query': q} for q in CODING_QUERIES] +
    [{'agent': 'search', 'query': q} for q in SEARCH_QUERIES]
)

print(f'{len(PROMPTS)} prompts loaded ({len(CODING_QUERIES)} coding, {len(SEARCH_QUERIES)} search)')

40 prompts loaded (20 coding, 20 search)


## 8 — Metrics + benchmark runner

In [15]:
import time, json
from dataclasses import dataclass, field, asdict
from typing import List
from openai import OpenAI
from tqdm.notebook import tqdm

@dataclass
class RequestResult:
    agent_type: str
    query_idx: int
    query: str
    ttft: float
    total_time: float
    cache_state: str

@dataclass
class BenchmarkResult:
    config_name: str
    requests: List[RequestResult] = field(default_factory=list)

    def summary(self):
        ttfts = [r.ttft for r in self.requests]
        by_state = {}
        for r in self.requests:
            by_state.setdefault(r.cache_state, []).append(r.ttft)
        print(f"\n{'='*45}")
        print(f'Config : {self.config_name}')
        print(f'Requests: {len(self.requests)}')
        print(f'Mean TTFT: {sum(ttfts)/len(ttfts)*1000:.1f} ms')
        for state in ['cold', 'warm', 'hot']:
            if state in by_state:
                vals = by_state[state]
                print(f'  {state:5s}: {sum(vals)/len(vals)*1000:7.1f} ms  (n={len(vals)})')
        print(f"{'='*45}\n")

    def save(self, path: str):
        with open(path, 'w') as f:
            json.dump({'config': self.config_name,
                       'requests': [asdict(r) for r in self.requests]}, f, indent=2)
        print(f'Saved: {path}')


def measure_ttft(client, model, messages, agent_type, query_idx, query, cache_state):
    t0 = time.perf_counter()
    first_token_time = None
    stream = client.chat.completions.create(
        model=model, messages=messages, max_tokens=256, stream=True
    )
    for chunk in stream:
        if (first_token_time is None
                and chunk.choices
                and chunk.choices[0].delta.content):
            first_token_time = time.perf_counter()
    total_time = time.perf_counter() - t0
    ttft = (first_token_time - t0) if first_token_time else total_time
    return RequestResult(
        agent_type=agent_type, query_idx=query_idx, query=query[:60],
        ttft=ttft, total_time=total_time, cache_state=cache_state
    )


def run_round_robin(config_name, base_url, model, num_rounds=3):
    client = OpenAI(base_url=base_url, api_key='none')
    result = BenchmarkResult(config_name=config_name)

    coding = [p for p in PROMPTS if p['agent'] == 'coding']
    search = [p for p in PROMPTS if p['agent'] == 'search']
    interleaved = [x for pair in zip(coding, search) for x in pair]

    for round_num in range(num_rounds):
        cache_state = ['cold', 'warm', 'hot'][min(round_num, 2)]
        print(f'\n--- Round {round_num + 1} ({cache_state}) ---')

        for i, prompt in enumerate(tqdm(interleaved)):
            r = measure_ttft(
                client=client, model=model,
                messages=[
                    {'role': 'system', 'content': AGENT_SYSTEMS[prompt['agent']]},
                    {'role': 'user',   'content': prompt['query']}
                ],
                agent_type=prompt['agent'], query_idx=i,
                query=prompt['query'], cache_state=cache_state
            )
            result.requests.append(r)
            print(f"  [{r.agent_type:6s}] TTFT={r.ttft*1000:6.1f}ms  {prompt['query'][:50]}...")

    return result

print('benchmark runner ready')

benchmark runner ready


## 9 — Run benchmark

In [ ]:
MODEL = 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4'
BASE_URL = f'{PUBLIC_URL}/v1'

result = run_round_robin(
    config_name='lmcache_disk',
    base_url=BASE_URL,
    model=MODEL,
    num_rounds=3
)

result.summary()
result.save('/content/results/results_lmcache_disk.json')


--- Round 1 (cold) ---


  0%|          | 0/40 [00:00<?, ?it/s]

  [coding] TTFT=2670.5ms  Implement a thread-safe LRU cache in Python with O...
  [search] TTFT= 413.1ms  What are the main architectural differences betwee...
  [coding] TTFT= 381.8ms  Write a Python context manager that retries a bloc...
  [search] TTFT= 393.1ms  Explain how RLHF differs from DPO in LLM fine-tuni...
  [coding] TTFT= 411.4ms  Design a rate limiter class using the token bucket...
  [search] TTFT= 481.1ms  What is the current state of quantum computing for...
  [coding] TTFT= 443.0ms  Implement a trie for autocomplete with insert and ...
  [search] TTFT= 388.2ms  Summarize the key ideas behind retrieval-augmented...
  [coding] TTFT= 428.3ms  Write a decorator that caches function results wit...
  [search] TTFT= 377.1ms  How does PagedAttention improve GPU memory efficie...
  [coding] TTFT= 404.1ms  Optimize this O(n^2) solution to find pairs summin...
  [search] TTFT= 410.7ms  What were the main contributions of the Attention ...
  [coding] TTFT= 401.5ms  Implement cons

  0%|          | 0/40 [00:00<?, ?it/s]

  [coding] TTFT= 378.9ms  Implement a thread-safe LRU cache in Python with O...
  [search] TTFT= 400.5ms  What are the main architectural differences betwee...
  [coding] TTFT= 399.9ms  Write a Python context manager that retries a bloc...
  [search] TTFT= 432.1ms  Explain how RLHF differs from DPO in LLM fine-tuni...
  [coding] TTFT= 392.3ms  Design a rate limiter class using the token bucket...
  [search] TTFT= 396.7ms  What is the current state of quantum computing for...
  [coding] TTFT= 380.7ms  Implement a trie for autocomplete with insert and ...
  [search] TTFT= 381.0ms  Summarize the key ideas behind retrieval-augmented...
  [coding] TTFT= 555.6ms  Write a decorator that caches function results wit...
  [search] TTFT= 397.5ms  How does PagedAttention improve GPU memory efficie...
  [coding] TTFT= 399.2ms  Optimize this O(n^2) solution to find pairs summin...
  [search] TTFT= 389.1ms  What were the main contributions of the Attention ...
  [coding] TTFT= 368.5ms  Implement cons

  0%|          | 0/40 [00:00<?, ?it/s]

  [coding] TTFT= 379.4ms  Implement a thread-safe LRU cache in Python with O...
  [search] TTFT= 398.9ms  What are the main architectural differences betwee...
  [coding] TTFT= 388.8ms  Write a Python context manager that retries a bloc...
  [search] TTFT= 377.8ms  Explain how RLHF differs from DPO in LLM fine-tuni...
  [coding] TTFT= 371.1ms  Design a rate limiter class using the token bucket...
  [search] TTFT= 387.7ms  What is the current state of quantum computing for...
  [coding] TTFT= 388.1ms  Implement a trie for autocomplete with insert and ...
  [search] TTFT= 382.6ms  Summarize the key ideas behind retrieval-augmented...
  [coding] TTFT= 554.7ms  Write a decorator that caches function results wit...
  [search] TTFT= 381.0ms  How does PagedAttention improve GPU memory efficie...
  [coding] TTFT= 386.7ms  Optimize this O(n^2) solution to find pairs summin...
  [search] TTFT= 374.6ms  What were the main contributions of the Attention ...
  [coding] TTFT= 366.2ms  Implement cons

## 10 — Plot results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

requests_data = result.requests
states = ['cold', 'warm', 'hot']
by_state = {}
for r in requests_data:
    by_state.setdefault(r.cache_state, []).append(r.ttft * 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

means = [sum(by_state.get(s, [0])) / max(len(by_state.get(s, [0])), 1) for s in states]
colors = ['#e74c3c', '#3498db', '#2ecc71']
axes[0].bar(states, means, color=colors, width=0.5)
axes[0].set_xlabel('Cache State')
axes[0].set_ylabel('Mean TTFT (ms)')
axes[0].set_title('Mean TTFT by Cache State — LMCache Disk')
axes[0].grid(axis='y', alpha=0.3)
for i, (s, m) in enumerate(zip(states, means)):
    axes[0].text(i, m + 10, f'{m:.0f}ms', ha='center', fontsize=10)

coding_pts = [(i, r.ttft * 1000) for i, r in enumerate(requests_data) if r.agent_type == 'coding']
search_pts = [(i, r.ttft * 1000) for i, r in enumerate(requests_data) if r.agent_type == 'search']
if coding_pts:
    axes[1].plot(*zip(*coding_pts), 'o-', label='Coding Agent', alpha=0.7, markersize=3)
if search_pts:
    axes[1].plot(*zip(*search_pts), 's-', label='Search Agent', alpha=0.7, markersize=3)
axes[1].set_xlabel('Request Index')
axes[1].set_ylabel('TTFT (ms)')
axes[1].set_title('TTFT Over Time — round 1=cold, 2=warm, 3=hot')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/results/component1_ttft.png', dpi=150)
plt.show()
print('Plot saved to /content/results/component1_ttft.png')

## 11 — Download results

In [ ]:
from google.colab import files
files.download('/content/results/results_lmcache_disk.json')
files.download('/content/results/component1_ttft.png')